# TweetAge v8: ordenes de caracteres altos y ensamble de modelos de lenguaje con EM

**Objetivo.** Dos ideas dentro de los modelos de lenguaje propios (ModifiedKN de v6 por clase), comparadas contra la referencia R en los mismos cinco pliegues.

1. **Ordenes altos (O6, O7).** Un historial de 5 o 6 caracteres captura secuencias mas largas (sufijos, emoticonos, alargamientos, abreviaturas) que el orden 4 corta. Kneser-Ney modificado reparte masa hacia los ordenes bajos cuando el historial largo es escaso, asi que subir el orden no deberia empeorar mucho la estimacion en contextos raros. Unico cambio respecto a R: el orden de caracteres.
2. **Ensamble con EM (E).** En lugar de elegir un orden, se mezclan varios: P(simbolo | historial) = suma_k w_k P_k(simbolo | historial), con pesos w_k estimados por EM sobre datos no vistos (interpolacion lineal de Jelinek y Mercer, 1980; EM de Dempster, Laird y Rubin, 1977). La mezcla puede tomar del orden alto cuando tiene evidencia y del bajo cuando no.

Ambas siguen siendo modelos de lenguaje propios y generativos: la clasificacion es por log-verosimilitud mas log prior, sin vectores de caracteristicas, embeddings, pesos preentrenados ni clasificador discriminativo sobre scores.

**Brazos.**
- R: ModifiedKN por clase, caracteres orden 4, palabras orden 3, score = 0,75 x caracteres + 0,25 x palabras + log prior. Se recalcula y se verifica contra F1 OOF 0,440081 y por pliegue 0,440843 / 0,432485 / 0,440697 / 0,447354 / 0,439014 (tolerancia 1e-6).
- O6: R con caracteres de orden 6.
- O7: R con caracteres de orden 7.
- E: siete ModifiedKN por clase, caracteres de orden 3, 4, 5 y 6, y palabras de orden 2, 3 y 4. Como una mezcla por token solo tiene sentido entre modelos que predicen el mismo tipo de simbolo, hay dos mezclas: una de caracteres (4 pesos que suman 1) y otra de palabras (3 pesos que suman 1). Los pesos se comparten entre clases y se estiman con EM sobre una validacion interna formada por el 10 % de los grupos del ajuste del pliegue, nunca sobre el pliegue de evaluacion. EM parte de pesos iguales, con un maximo de 50 iteraciones o convergencia de 1e-6 en los pesos. Despues los modelos se reentrenan con todo el ajuste del pliegue y se aplican al pliegue de evaluacion. score = 0,75 x log P_mezcla(caracteres) + 0,25 x log P_mezcla(palabras) + log prior, la misma combinacion entre niveles que R. Sin calibracion de sesgos.

**Detalle de implementacion.** Un solo ModifiedKN por clase y nivel, de orden maximo 7 en caracteres y 4 en palabras, sirve para todos los ordenes inferiores: la prueba de conteos compartidos de v6 comprueba que da los mismos log P que entrenar cada orden por separado.

**Regla de decision (fijada antes de ejecutar).** Un brazo se acepta solo si la media de las cinco diferencias pareadas de F1 macro contra R es > 0 y mayor que su desviacion estandar muestral (ddof=1). Entre aceptados se elige la mayor diferencia media, con desempate en el orden O6, O7, E. Si ninguno se acepta: "NO PASA", sin holdout ni CSV. Si se acepta: holdout evaluado una sola vez y `age_submission_v8_ordenes_y_ensamble.csv` con escritura exclusiva. Heuristica, no prueba formal.


In [1]:
import ast
import csv
import gc
import hashlib
import html
import json
import math
import platform
import re
import time
from collections import Counter
from functools import lru_cache
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from IPython.display import display, Markdown

SEED=42
np.random.seed(SEED)
CLASSES=np.array(['13-17','18-24','25-34','35-49','50-64','65-xx'])
BASE_EPSILON=1e-6
BOS,EOS,UNK='\x02','\x03','\x01'
CHAR_ORDERS=[3,4,5,6,7]
WORD_ORDERS=[2,3,4]
CHAR_WEIGHT=0.75
ARMS=['R','O6','O7','E']
ARM_CHAR_ORDER={'R':4,'O6':6,'O7':7}
ARM_WORD_ORDER=3
E_CHAR_ORDERS=[3,4,5,6]
E_WORD_ORDERS=[2,3,4]
EM_MAX_ITERATIONS=50
EM_TOLERANCE=1e-6
INTERNAL_VALIDATION_FRACTION=0.1
REF_FOLD_F1=np.array([0.440843,0.432485,0.440697,0.447354,0.439014])
REF_OOF_F1=0.440081
REF_TOLERANCE=1e-6
NORMALIZATION_TOLERANCE=1e-9
roots=[Path.cwd(),Path.cwd()/'workshops'/'Kaggle-tweet-age',Path.cwd()/'natural-language-processing-workshops'/'workshops'/'Kaggle-tweet-age']
OUT_DIR=next((candidate for candidate in roots if (candidate/'TweetAge_v8_ordenes_y_ensamble.ipynb').is_file()),None)
assert OUT_DIR is not None,'Ejecute desde la carpeta del notebook o la raiz del repo.'
NOTEBOOK_PATH=OUT_DIR/'TweetAge_v8_ordenes_y_ensamble.ipynb'
SUB_PATH=OUT_DIR/'age_submission_v8_ordenes_y_ensamble.csv'
SPLIT_PATH=OUT_DIR/'split_idx_v7_lower_seed42.npz'
DATA_DIR=OUT_DIR/'tweet-age-2026-puj'
assert not SUB_PATH.exists(),'El CSV ya existe; no se sobrescribe.'
INPUT_HASHES={name:hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest() for name in ['age_train.csv','age_test.csv','age_submission.csv']}
PROTECTED_HASHES={path.name:hashlib.sha256(path.read_bytes()).hexdigest() for path in OUT_DIR.iterdir() if path.is_file() and path!=NOTEBOOK_PATH}
assert PROTECTED_HASHES[SPLIT_PATH.name]=='692fcfd2fc5a8e1ead91763d4741da5d14f86fecd201b2e419c5e7d47c16be83','Hash del NPZ v7_lower distinto; detener.'
print('Python',platform.python_version(),'numpy',np.__version__,'pandas',pd.__version__,'sklearn',sklearn.__version__)
print('Semilla',SEED,'| notebook',NOTEBOOK_PATH.name,'| brazos',ARMS)
print('SHA-256 NPZ v7_lower:',PROTECTED_HASHES[SPLIT_PATH.name])
print('SHA-256 age_train.csv:',INPUT_HASHES['age_train.csv'])

def check_compliance(notebook):
    # Guardia AST de v6 mas prohibiciones de esta ronda: vectorizadores, NB de sklearn, matrices dispersas, embeddings, preentrenados, torch y clasificadores discriminativos.
    forbidden = {'tfidfvectorizer', 'countvectorizer', 'hashingvectorizer', 'dictvectorizer', 'feature_extraction', 'linear_model', 'transformers', 'transformer', 'embedding', 'embeddings', 'word2vec', 'glove', 'gensim', 'pretrained', 'from_pretrained', 'multinomialnb', 'complementnb', 'bernoullinb', 'gaussiannb', 'naive_bayes', 'sparse', 'csr_matrix', 'csc_matrix', 'coo_matrix', 'lil_matrix', 'torch', 'svm', 'svc', 'linearsvc', 'ensemble', 'tree', 'neighbors', 'neural_network', 'mlpclassifier', 'logisticregression', 'sgdclassifier', 'discriminant_analysis', 'xgboost', 'lightgbm', 'catboost'}
    problems = []
    for cell_number, cell in enumerate(notebook['cells'], 1):
        if cell['cell_type'] != 'code':
            continue
        source = ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
        for node in ast.walk(ast.parse(source)):
            names = []
            if isinstance(node, ast.Name):
                names = [node.id]
            elif isinstance(node, ast.Attribute):
                names = [node.attr]
            elif isinstance(node, ast.Import):
                names = [alias.name for alias in node.names]
            elif isinstance(node, ast.ImportFrom):
                names = [node.module or ''] + [alias.name for alias in node.names]
            for name in names:
                if any(part.lower() in forbidden for part in name.split('.')):
                    problems.append((cell_number, node.lineno, name))
            if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in {'eval', 'exec', '__import__'}:
                problems.append((cell_number, node.lineno, 'ejecucion dinamica'))
    if problems:
        raise RuntimeError(f'Guardia AST: codigo prohibido {problems}')
    return True

with NOTEBOOK_PATH.open(encoding='utf-8') as handle:
    assert check_compliance(json.load(handle))
print('Guardia AST de v6 superada antes de entrenar; sin vectorizadores, NB de sklearn, matrices de conteos, embeddings, preentrenados, torch ni clasificadores discriminativos.')


Python 3.12.14 numpy 2.5.4 pandas 3.0.6 sklearn 1.9.1
Semilla 42 | notebook TweetAge_v8_ordenes_y_ensamble.ipynb | brazos ['R', 'O6', 'O7', 'E']
SHA-256 NPZ v7_lower: 692fcfd2fc5a8e1ead91763d4741da5d14f86fecd201b2e419c5e7d47c16be83
SHA-256 age_train.csv: 39c6c8270366fe155f3c8dd232e31867bee659af8908121d04b6bd89bd163748
Guardia AST de v6 superada antes de entrenar; sin vectorizadores, NB de sklearn, matrices de conteos, embeddings, preentrenados, torch ni clasificadores discriminativos.


## Datos, normalizacion A y verificacion del NPZ v7_lower
Celda identica a la de v7 hiperparametros: hashes del train y de los grupos, cero fuga en minusculas en holdout y en los cinco pliegues.

In [2]:
def verify_csv(path, columns, delimiter, bom):
    with path.open('rb') as handle:
        prefix = handle.read(32768)
    first_line = prefix.decode('utf-8-sig').splitlines()[0]
    actual = csv.Sniffer().sniff(first_line, delimiters=',;\t|').delimiter
    header = next(csv.reader([first_line], delimiter=actual))
    assert actual == delimiter and prefix.startswith(b'\xef\xbb\xbf') == bom, (path.name, actual)
    assert header == columns, (path.name, header)
    print(path.name, 'BOM=', bom, 'separador=', repr(actual), 'columnas=', header)
    return header

URL_RE = re.compile(r'https?://[^\s]+|www\.[^\s]+', re.IGNORECASE)

MENTION_RE = re.compile(r'@\w+', re.UNICODE)

REPEAT_RE = re.compile(r'([^\W\d_])\1{3,}', re.UNICODE)

def replace_url(match):
    value = match.group(0)
    core = value.rstrip('.,!?;:)]}')
    return '<url>' + value[len(core):]

def normalize_text(text):
    return REPEAT_RE.sub(lambda match: match.group(1) * 3, MENTION_RE.sub('<user>', URL_RE.sub(replace_url, text)))

verify_csv(DATA_DIR/'age_train.csv',['id','text','age_range'],';',True)
reference_columns=verify_csv(DATA_DIR/'age_submission.csv',['id','age_range'],',',False)
train_raw=pd.read_csv(DATA_DIR/'age_train.csv',sep=';',encoding='utf-8-sig',dtype=str,keep_default_na=False)
assert list(train_raw.columns)==['id','text','age_range']
clean=train_raw.text.fillna('').str.strip().map(html.unescape)
excluded=clean.eq('')|clean.str.contains('#\u00bfNOMBRE',regex=False)
train=train_raw.loc[~excluded].copy()
train['raw_row']=np.flatnonzero(~excluded.to_numpy())
train['text']=clean.loc[~excluded]
train=train.reset_index(drop=True)
assert train.id.is_unique and set(train.age_range)==set(CLASSES)
train['normalized']=train.text.map(normalize_text)
y_all=pd.Categorical(train.age_range,categories=CLASSES).codes.astype(np.int64)
with np.load(SPLIT_PATH,allow_pickle=False) as saved:
    split={name:saved[name].copy() for name in saved.files}
assert str(split['train_sha256'])==INPUT_HASHES['age_train.csv'] and int(split['seed'])==SEED
assert np.array_equal(split['retained_raw_idx'],train.raw_row.to_numpy())
group_keys=np.array([text.lower() for text in train.normalized],dtype=object)
groups_hash=hashlib.sha256(json.dumps(group_keys.tolist(),ensure_ascii=False,separators=(',',':')).encode('utf-8')).hexdigest()
assert groups_hash==str(split['groups_sha256'])
group_ids,unique_keys=pd.factorize(group_keys,sort=True)
assert np.array_equal(group_ids,split['group_ids'])
idx_development,idx_holdout=split['train_idx'],split['valid_idx']
dev_groups,holdout_groups=split['train_groups'],split['valid_groups']
assert np.array_equal(np.sort(np.concatenate([idx_development,idx_holdout])),np.arange(len(train)))
assert np.array_equal(train.raw_row.to_numpy()[idx_development],split['raw_train_idx'])
assert np.array_equal(train.raw_row.to_numpy()[idx_holdout],split['raw_valid_idx'])
assert set(group_ids[idx_development])==set(dev_groups) and set(group_ids[idx_holdout])==set(holdout_groups)
assert not np.intersect1d(dev_groups,holdout_groups).size
assert not set(group_keys[idx_development])&set(group_keys[idx_holdout])
group_counts=np.zeros((len(unique_keys),6),dtype=np.int64)
np.add.at(group_counts,(group_ids,y_all),1)
group_targets=group_counts.argmax(axis=1)
texts_development=train.normalized.iloc[idx_development].to_numpy()
y_development=y_all[idx_development]
groups_development=group_ids[idx_development]
group_folds=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(dev_groups,group_targets[dev_groups]))
folds=[]
leak_rows=[dict(validacion='holdout',textos_compartidos=0)]
for fold_number,(fit_positions,app_positions) in enumerate(group_folds,1):
    fit_idx=np.flatnonzero(np.isin(groups_development,dev_groups[fit_positions]))
    app_idx=np.flatnonzero(np.isin(groups_development,dev_groups[app_positions]))
    shared={text.lower() for text in texts_development[fit_idx]}&{text.lower() for text in texts_development[app_idx]}
    leak_rows.append(dict(validacion=str(fold_number),textos_compartidos=len(shared)))
    if shared:
        print('DETENER: fuga en pliegue',fold_number,'ejemplos',sorted(shared)[:10])
        raise RuntimeError('Fuga en minusculas; no entrenar.')
    folds.append((fit_idx,app_idx))
print('NPZ REUTILIZADO SIN REGENERAR:',SPLIT_PATH.name)
print('SHA-256 train y grupos correctos. SHA-256 grupos:',groups_hash)
print('Filas train original/limpio/excluidas:',len(train_raw),len(train),int(excluded.sum()))
print('Desarrollo grupos/filas:',len(dev_groups),len(idx_development),'holdout grupos/filas:',len(holdout_groups),len(idx_holdout))
print('Textos solo URL:',int(train.normalized.str.strip().eq('<url>').sum()),'solo usuario:',int(train.normalized.str.strip().eq('<user>').sum()))
display(pd.DataFrame(leak_rows))


age_train.csv BOM= True separador= ';' columnas= ['id', 'text', 'age_range']
age_submission.csv BOM= False separador= ',' columnas= ['id', 'age_range']


NPZ REUTILIZADO SIN REGENERAR: split_idx_v7_lower_seed42.npz
SHA-256 train y grupos correctos. SHA-256 grupos: f482fe9f4a7495a0050b857b8eda3b3bc9177c31e2ae6f10c9138cfe8df2b1a8
Filas train original/limpio/excluidas: 169336 169181 155
Desarrollo grupos/filas: 126309 132987 holdout grupos/filas: 31184 36194
Textos solo URL: 3785 solo usuario: 287


,validacion,textos_compartidos
0,holdout,0
1,1,0
2,2,0
3,3,0
4,4,0
5,5,0


## Kneser-Ney modificado de v6
Incluye la prueba de conteos compartidos para los ordenes 3-7 de caracteres y 2-4 de palabras usados aqui.

In [3]:
TOKEN_RE = re.compile(r'<url>|<user>|\w+|[^\w\s]', re.UNICODE)

def tokenize(text, level):
    return list(text) if level == 'char' else TOKEN_RE.findall(text.lower())

def make_encoder(texts, level):
    frequencies = Counter()
    for text in texts:
        frequencies.update(tokenize(text, level))
    cutoff = 1 if level == 'char' else 2
    return {token: chr(index + 256) for index, token in enumerate(sorted(token for token, count in frequencies.items() if count >= cutoff))}

def encode_text(text, level, mapping):
    return ''.join(mapping.get(token, UNK) for token in tokenize(text, level))

def estimate_discounts(counts):
    histogram = Counter(counts.values())
    count1, count2, count3, count4 = [histogram.get(value, 0) for value in [1, 2, 3, 4]]
    factor = count1 / (count1 + 2 * count2) if count1 + 2 * count2 else 0.0
    first = 1 - 2 * factor * count2 / count1 if count1 and count2 else 0.75
    second = 2 - 3 * factor * count3 / count2 if count1 and count2 else 0.75
    third = 3 - 4 * factor * count4 / count3 if count1 and count2 and count3 else 0.75
    values = tuple(float(min(bucket - 1e-6, max(1e-6, value))) for bucket, value in enumerate([first, second, third], 1))
    return values, dict(N1=count1, N2=count2, N3=count3, N4=count4)

def discount_for(count, discounts):
    return discounts[min(count, 3) - 1] if count > 0 else 0.0

class ModifiedKN:
    def __init__(self, max_order, support):
        self.max_order = max_order
        self.support = set(support) | {EOS, UNK}

    def fit(self, sequences):
        highest = max(2, self.max_order)
        self.raw = {order: Counter() for order in range(2, highest + 1)}
        for sequence in sequences:
            padded = BOS * (highest - 1) + sequence + EOS
            for order, counts in self.raw.items():
                counts.update(padded[position - order + 1:position + 1] for position in range(highest - 1, len(padded)))
        self.continuation = {}
        for order in range(1, highest):
            effective = Counter()
            for gram in self.raw[order + 1]:
                effective[gram[1:]] += 1
            self.continuation[order] = effective
        self.base = self.continuation[1]
        self.base_total = sum(self.base.values()) + BASE_EPSILON * len(self.support)
        self.discount_records = []
        def prepare(count_tables, origin):
            discounts_by_order, histories_by_order = {}, {}
            for order, counts in count_tables.items():
                if order == 1:
                    continue
                discounts, histogram = estimate_discounts(counts)
                discounts_by_order[order] = discounts
                totals, removed = Counter(), Counter()
                for gram, count in counts.items():
                    totals[gram[:-1]] += count
                    removed[gram[:-1]] += discount_for(count, discounts)
                histories_by_order[order] = {history: (total, removed[history]) for history, total in totals.items()}
                self.discount_records.append(dict(origen=origin, orden=order, D1=discounts[0], D2=discounts[1], D3=discounts[2], **histogram))
            return discounts_by_order, histories_by_order
        self.raw_discounts, self.raw_hist = prepare(self.raw, 'frecuencias')
        self.cont_discounts, self.cont_hist = prepare(self.continuation, 'continuaciones')
        self.lower_cached = lru_cache(maxsize=200000)(self._lower)
        self.log_cached = lru_cache(maxsize=200000)(self._log_gram)
        return self

    def _lower(self, gram):
        if len(gram) == 1:
            return (self.base.get(gram, 0) + BASE_EPSILON) / self.base_total
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.cont_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.continuation[order].get(gram, 0)
        return (count - discount_for(count, self.cont_discounts[order]) + removed * lower) / total

    def probability(self, gram):
        if len(gram) == 1:
            return self.lower_cached(gram)
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.raw_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.raw[order].get(gram, 0)
        return (count - discount_for(count, self.raw_discounts[order]) + removed * lower) / total

    def _log_gram(self, gram):
        probability = self.probability(gram)
        if not 0 < probability <= 1 + 1e-10:
            raise ArithmeticError((repr(gram), probability))
        return math.log(probability)

    def log_probability(self, sequence, order):
        assert 1 <= order <= self.max_order
        padded = BOS * (order - 1) + sequence + EOS
        grams = Counter(padded[position - order + 1:position + 1] for position in range(order - 1, len(padded)))
        return sum(count * self.log_cached(gram) for gram, count in grams.items())

    def close(self):
        self.lower_cached.cache_clear()
        self.log_cached.cache_clear()

def test_modified_kn():
    values, _ = estimate_discounts(Counter({'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 1, 'f': 2, 'g': 3, 'h': 1}))
    assert len(set(values)) == 3 and all(0 < value < bucket for bucket, value in enumerate(values, 1))
    model = ModifiedKN(4, set('abc')).fit(['ab', 'ab', 'ac', 'bc', 'abc', 'abc', 'abc'])
    assert model.base['a'] == 1 and model.raw[2][BOS + 'a'] == 6
    for order in [1, 2, 3, 4]:
        contexts = set(model.raw_hist.get(order, {})) | {'', 'zz', BOS * (order - 1)}
        for context in contexts:
            history = (BOS * (order - 1) + context)[-(order - 1):] if order > 1 else ''
            assert abs(sum(model.probability(history + symbol) for symbol in model.support) - 1) < 1e-9, (order, repr(history))
        assert math.isfinite(model.log_probability('a' + UNK, order))
    assert model.log_probability('ab', 4) > model.log_probability('cc', 4)
    model.close()
    assert tokenize('Hola HOLA', 'word') == ['hola', 'hola']
    assert tokenize('Hola', 'char')[0] == 'H'
    print('Pruebas MKN: tres descuentos, normalizacion, continuaciones, BOS/EOS, UNK y capitalizacion correctos.')

def test_shared_counts():
    toy_texts=['Hola hola!','Hola amiga!','Adios hola!','Hola hola!','Amiga adios!']
    for level,orders in [('char',CHAR_ORDERS),('word',WORD_ORDERS)]:
        mapping=make_encoder(toy_texts,level)
        sequences=[encode_text(text,level,mapping) for text in toy_texts]
        support=set(mapping.values())
        shared_model=ModifiedKN(max(orders),support).fit(sequences)
        for order in orders:
            separate_model=ModifiedKN(order,support).fit(sequences)
            for sequence in sequences+[UNK]:
                assert abs(shared_model.log_probability(sequence,order)-separate_model.log_probability(sequence,order))<1e-12,(level,order)
            separate_model.close()
        shared_model.close()
    print('Prueba de conteos compartidos: mismos log P que entrenar cada orden por separado.')

test_modified_kn()
test_shared_counts()
timing_rows=[]

def fit_score_orders(fit_texts,fit_labels,app_texts,level,orders,phase,fold):
    started=time.perf_counter()
    mapping=make_encoder(fit_texts,level)
    encoded_fit=[encode_text(text,level,mapping) for text in fit_texts]
    encoded_app=[encode_text(text,level,mapping) for text in app_texts]
    preprocessing=time.perf_counter()-started
    matrices={order:np.zeros((len(app_texts),6)) for order in orders}
    fit_seconds=0.0
    score_seconds={order:0.0 for order in orders}
    for class_index in range(6):
        fit_started=time.perf_counter()
        model=ModifiedKN(max(orders),set(mapping.values())).fit(sequence for sequence,target in zip(encoded_fit,fit_labels) if target==class_index)
        fit_seconds+=time.perf_counter()-fit_started
        for order in orders:
            score_started=time.perf_counter()
            matrices[order][:,class_index]=[model.log_probability(sequence,order) for sequence in encoded_app]
            score_seconds[order]+=time.perf_counter()-score_started
        model.close()
        del model
        gc.collect()
    for order in orders:
        timing_rows.append(dict(fase=phase,pliegue=fold,nivel=level,orden=order,ajuste_compartido_s=preprocessing+fit_seconds,puntuacion_s=score_seconds[order],tiempo_amortizado_s=score_seconds[order]+(preprocessing+fit_seconds)/len(orders)))
    print(phase,fold,level,'ordenes',orders,'segundos',round(time.perf_counter()-started,2),flush=True)
    return matrices


Pruebas MKN: tres descuentos, normalizacion, continuaciones, BOS/EOS, UNK y capitalizacion correctos.
Prueba de conteos compartidos: mismos log P que entrenar cada orden por separado.


## Puntuacion por posiciones, EM y pruebas

In [4]:
def gram_positions(encoded,max_order,orders):
    # Para cada posicion (texto + EOS) y cada orden k: indice del k-grama distinto (ultimos k simbolos con relleno BOS).
    padding=BOS*(max_order-1)
    rows=[]
    indices={order:({},[]) for order in orders}
    for row,sequence in enumerate(encoded):
        padded=padding+sequence+EOS
        for end in range(max_order-1,len(padded)):
            rows.append(row)
            for order in orders:
                lookup,positions=indices[order]
                positions.append(lookup.setdefault(padded[end-order+1:end+1],len(lookup)))
    return np.array(rows,dtype=np.int64),{order:(list(lookup),np.array(positions,dtype=np.int64)) for order,(lookup,positions) in indices.items()}

def gram_probabilities(model,grams):
    values=np.array([model.probability(gram) for gram in grams])
    if len(values) and not ((values>0).all() and (values<=1+1e-10).all()):
        raise ArithmeticError('Probabilidad fuera de (0,1].')
    return values

def em_weights(probabilities):
    # EM de pesos de mezcla sobre tokens no vistos: probabilities (tokens x modelos).
    count=probabilities.shape[1]
    weights=np.full(count,1/count)
    trace=[]
    for iteration in range(1,EM_MAX_ITERATIONS+1):
        weighted=probabilities*weights
        mixture=weighted.sum(axis=1,keepdims=True)
        new_weights=(weighted/mixture).mean(axis=0)
        change=float(np.abs(new_weights-weights).max())
        weights=new_weights
        trace.append(dict(iteracion=iteration,cambio_max=change,logver_media=float(np.log(mixture).mean())))
        if change<EM_TOLERANCE:
            break
    assert abs(weights.sum()-1)<1e-12 and (weights>=0).all(),weights
    return weights,dict(iteraciones=iteration,cambio_final=change,logver_media_final=float(np.log((probabilities*weights).sum(axis=1)).mean()))

def internal_split(groups,seed):
    unique_groups=np.unique(groups)
    rng=np.random.default_rng(seed)
    held=rng.choice(unique_groups,size=int(round(INTERNAL_VALIDATION_FRACTION*len(unique_groups))),replace=False)
    return np.isin(groups,held)

def heldout_token_probabilities(inner_texts,inner_labels,val_texts,val_labels,level,orders):
    # Probabilidades por token de cada orden, con el modelo de la clase verdadera, entrenado solo con la parte interna del ajuste.
    mapping=make_encoder(inner_texts,level)
    encoded_inner=[encode_text(text,level,mapping) for text in inner_texts]
    encoded_val=[encode_text(text,level,mapping) for text in val_texts]
    rows,indices=gram_positions(encoded_val,max(orders),orders)
    token_class=val_labels[rows]
    table=np.zeros((len(rows),len(orders)))
    for class_index in range(6):
        model=ModifiedKN(max(orders),set(mapping.values())).fit(sequence for sequence,label in zip(encoded_inner,inner_labels) if label==class_index)
        mask=token_class==class_index
        for column,order in enumerate(orders):
            grams,positions=indices[order]
            needed=np.unique(positions[mask])
            values=np.zeros(len(grams))
            values[needed]=gram_probabilities(model,[grams[index] for index in needed])
            table[mask,column]=values[positions[mask]]
        model.close()
        del model
        gc.collect()
    return table

def normalization_rows(model,order,label,level):
    symbols=sorted(model.support)
    histories=[BOS*(order-1)]
    histories+=[history for history,_ in sorted(model.raw_hist[order].items(),key=lambda item:-item[1][0])[:2]]
    histories+=[sorted(model.raw_hist[order])[len(model.raw_hist[order])//2],UNK*(order-1)]
    rows=[]
    for history in dict.fromkeys(histories):
        mass=sum(model.probability(history+symbol) for symbol in symbols)
        rows.append(dict(nivel=level,orden=order,clase=label,historial=repr(history),visto=history in model.raw_hist[order],suma=mass,error=abs(mass-1)))
    return rows

def full_level_scores(fit_texts,fit_labels,app_texts,level,max_order,single_orders,mix_orders,mix_weights,check_orders=()):
    # Entrena un ModifiedKN por clase de orden max_order con todo el ajuste y devuelve log P por orden y log P de la mezcla.
    mapping=make_encoder(fit_texts,level)
    encoded_fit=[encode_text(text,level,mapping) for text in fit_texts]
    encoded_app=[encode_text(text,level,mapping) for text in app_texts]
    orders=sorted(set(single_orders)|set(mix_orders))
    rows,indices=gram_positions(encoded_app,max_order,orders)
    single={order:np.zeros((len(app_texts),6)) for order in single_orders}
    mixture=np.zeros((len(app_texts),6))
    checks=[]
    for class_index in range(6):
        model=ModifiedKN(max_order,set(mapping.values())).fit(sequence for sequence,label in zip(encoded_fit,fit_labels) if label==class_index)
        per_position={}
        for order in orders:
            grams,positions=indices[order]
            per_position[order]=gram_probabilities(model,grams)[positions]
        for order in single_orders:
            single[order][:,class_index]=np.bincount(rows,weights=np.log(per_position[order]),minlength=len(app_texts))
        mixed=sum(weight*per_position[order] for weight,order in zip(mix_weights,mix_orders))
        mixture[:,class_index]=np.bincount(rows,weights=np.log(mixed),minlength=len(app_texts))
        if class_index in (0,5):
            for order in check_orders:
                checks.extend(normalization_rows(model,order,CLASSES[class_index],level))
        model.close()
        del model,per_position
        gc.collect()
    return single,mixture,checks

def arm_scores(fit_texts,fit_labels,fit_groups,app_texts,seed,check_normalization=False):
    started=time.perf_counter()
    held=internal_split(fit_groups,seed)
    assert not {text.lower() for text in fit_texts[held]}&{text.lower() for text in fit_texts[~held]}
    weights,em_info={},{}
    for level,orders in [('char',E_CHAR_ORDERS),('word',E_WORD_ORDERS)]:
        table=heldout_token_probabilities(fit_texts[~held],fit_labels[~held],fit_texts[held],fit_labels[held],level,orders)
        weights[level],em_info[level]=em_weights(table)
        em_info[level]['tokens_validacion']=len(table)
    em_seconds=time.perf_counter()-started
    characters,char_mixture,checks=full_level_scores(fit_texts,fit_labels,app_texts,'char',max(CHAR_ORDERS),sorted(ARM_CHAR_ORDER.values()),E_CHAR_ORDERS,weights['char'],check_orders=(6,7) if check_normalization else ())
    words,word_mixture,_=full_level_scores(fit_texts,fit_labels,app_texts,'word',max(WORD_ORDERS),[ARM_WORD_ORDER],E_WORD_ORDERS,weights['word'])
    prior=np.log(np.bincount(fit_labels,minlength=6)/len(fit_labels))
    scores={arm:CHAR_WEIGHT*characters[order]+(1-CHAR_WEIGHT)*words[ARM_WORD_ORDER]+prior for arm,order in ARM_CHAR_ORDER.items()}
    scores['E']=CHAR_WEIGHT*char_mixture+(1-CHAR_WEIGHT)*word_mixture+prior
    info=dict(weights=weights,em=em_info,checks=checks,filas_validacion_interna=int(held.sum()),segundos_em=em_seconds,segundos=time.perf_counter()-started)
    return scores,info

def test_em_and_positions():
    # EM recupera pesos conocidos de una mezcla sintetica y el indexado por posiciones reproduce log_probability del ModifiedKN.
    rng=np.random.default_rng(SEED)
    component=rng.choice(3,size=20000,p=[0.6,0.3,0.1])
    probabilities=np.full((20000,3),0.01)
    probabilities[np.arange(20000),component]=0.9
    weights,info=em_weights(probabilities)
    assert abs(weights.sum()-1)<1e-12 and np.abs(weights-[0.6,0.3,0.1]).max()<0.02,weights
    texts=['Hola hola!','Hola amiga!','Adios hola!','Hola hola!','Amiga adios!']
    for level,orders in [('char',CHAR_ORDERS),('word',WORD_ORDERS)]:
        mapping=make_encoder(texts,level)
        encoded=[encode_text(text,level,mapping) for text in texts]
        model=ModifiedKN(max(orders),set(mapping.values())).fit(encoded)
        rows,indices=gram_positions(encoded+[UNK],max(orders),orders)
        for order in orders:
            grams,positions=indices[order]
            vectorized=np.bincount(rows,weights=np.log(gram_probabilities(model,grams)[positions]),minlength=len(encoded)+1)
            direct=[model.log_probability(sequence,order) for sequence in encoded+[UNK]]
            assert np.abs(vectorized-direct).max()<1e-9,(level,order)
        model.close()
    print('Prueba EM: recupera pesos 0,6/0,3/0,1 (error max',round(float(np.abs(weights-[0.6,0.3,0.1]).max()),4),'en',info['iteraciones'],'iteraciones), suman 1 y no negativos.')
    print('Prueba de posiciones: log P vectorizado igual a ModifiedKN.log_probability en todos los ordenes y niveles.')

test_em_and_positions()


Prueba EM: recupera pesos 0,6/0,3/0,1 (error max 0.0083 en 7 iteraciones), suman 1 y no negativos.
Prueba de posiciones: log P vectorizado igual a ModifiedKN.log_probability en todos los ordenes y niveles.


## Historial de intentos

In [5]:
history=pd.DataFrame([
    dict(intento='v5 KN (particion con fuga)',f1_oof='0,43847 (calibrado)',estado='Kaggle 0,45643; holdout inflado'),
    dict(intento='v6 KN modificado (agrupada)',f1_oof='0,44106 sin calibrar / 0,44353 calibrado',estado='Valida publicada; Kaggle 0,45737'),
    dict(intento='Referencia en v7_lower',f1_oof='0,440081',estado='Referencia de v7 y v8'),
    dict(intento='v7 ordinal',f1_oof='sin salida guardada',estado='NO PASA'),
    dict(intento='v7 descuento con concentracion',f1_oof='0,440875',estado='NO PASA'),
    dict(intento='v7 normalizacion (A/B/C)',f1_oof='0,440081 / 0,439709 / 0,436893',estado='NO PASA'),
    dict(intento='v7 hiperparametros (mejor C7)',f1_oof='0,437677',estado='NO PASA'),
    dict(intento='v7 LM neuronal GRU',f1_oof='pliegue 1: 0,411131',estado='NO PASA EN ETAPA 1'),
    dict(intento='v8 Naive Bayes',f1_oof='0,376727',estado='NO PASA'),
    dict(intento='v8 PPM-C',f1_oof='0,421730',estado='NO PASA'),
    dict(intento='v8 LM jerarquico (beta 10)',f1_oof='0,414583 (diferencia pareada -0,0255)',estado='NO PASA'),
    dict(intento='v8 ordenes y ensamble',f1_oof='(se llena al final)',estado='(se llena al final)'),
])
print('Historial de intentos (F1 macro OOF sin calibrar salvo indicacion; cifras solo de valores confirmados o con salida guardada). Las particiones de v5, v6 y v7_lower son distintas: la comparacion no es directa.')
display(history)


Historial de intentos (F1 macro OOF sin calibrar salvo indicacion; cifras solo de valores confirmados o con salida guardada). Las particiones de v5, v6 y v7_lower son distintas: la comparacion no es directa.


,intento,f1_oof,estado
0,v5 KN (particion con fuga),"0,43847 (calibrado)","Kaggle 0,45643; holdout inflado"
1,v6 KN modificado (agrupada),"0,44106 sin calibrar / 0,44353 calibrado","Valida publicada; Kaggle 0,45737"
2,Referencia en v7_lower,"0,440081",Referencia de v7 y v8
3,v7 ordinal,sin salida guardada,NO PASA
4,v7 descuento con concentracion,"0,440875",NO PASA
5,v7 normalizacion (A/B/C),"0,440081 / 0,439709 / 0,436893",NO PASA
6,v7 hiperparametros (mejor C7),"0,437677",NO PASA
7,v7 LM neuronal GRU,"pliegue 1: 0,411131",NO PASA EN ETAPA 1
8,v8 Naive Bayes,"0,376727",NO PASA
9,v8 PPM-C,"0,421730",NO PASA


## Validacion cruzada: R, O6, O7 y E

In [6]:
def macro_f1(y_true,y_pred):
    return f1_score(y_true,y_pred,labels=np.arange(6),average='macro',zero_division=0)

cv_started=time.perf_counter()
oof={arm:np.zeros((len(y_development),6)) for arm in ARMS}
fold_rows,weight_rows,em_rows,normalization_checks=[],[],[],[]
for fold_number,(fit_idx,app_idx) in enumerate(folds,1):
    fit_texts,app_texts=texts_development[fit_idx],texts_development[app_idx]
    assert not {text.lower() for text in fit_texts}&{text.lower() for text in app_texts}
    scores,info=arm_scores(fit_texts,y_development[fit_idx],groups_development[fit_idx],app_texts,seed=SEED+fold_number,check_normalization=fold_number==1)
    y_app=y_development[app_idx]
    for arm in ARMS:
        oof[arm][app_idx]=scores[arm]
    fold_rows.append(dict(pliegue=fold_number,**{f'f1_{arm}':macro_f1(y_app,scores[arm].argmax(axis=1)) for arm in ARMS},segundos=info['segundos'],segundos_em=info['segundos_em'],filas_validacion_interna=info['filas_validacion_interna']))
    for level,orders in [('char',E_CHAR_ORDERS),('word',E_WORD_ORDERS)]:
        weights=info['weights'][level]
        assert abs(weights.sum()-1)<1e-12 and (weights>=0).all()
        weight_rows.extend(dict(pliegue=fold_number,nivel=level,orden=order,peso=float(weight)) for order,weight in zip(orders,weights))
        em_rows.append(dict(pliegue=fold_number,nivel=level,**info['em'][level],suma_pesos=float(weights.sum()),min_peso=float(weights.min())))
    normalization_checks.extend(info['checks'])
    print(f'pliegue {fold_number}: '+' | '.join(f'{arm} {fold_rows[-1]["f1_"+arm]:.6f}' for arm in ARMS)+f' | pesos car {np.round(info["weights"]["char"],3).tolist()} pal {np.round(info["weights"]["word"],3).tolist()} | {info["segundos"]:.0f} s',flush=True)
    del scores
    gc.collect()

normalization_table=pd.DataFrame(normalization_checks)
print('VERIFICACION DE NORMALIZACION (pliegue 1, clases 13-17 y 65-xx, caracteres de orden 6 y 7):',len(normalization_table),'historiales; error maximo',float(normalization_table.error.max()))
display(normalization_table)
assert set(normalization_table.orden)=={6,7} and normalization_table.error.max()<NORMALIZATION_TOLERANCE
print('NORMALIZACION: PASA, los modelos de orden 6 y 7 suman 1 sobre el vocabulario con tolerancia 1e-9.')

em_table=pd.DataFrame(em_rows)
print('EM por pliegue y nivel: los pesos suman 1 y son no negativos en todos los pliegues.')
display(em_table)
weights_table=pd.DataFrame(weight_rows)
weights_pivot=weights_table.pivot_table(index=['nivel','orden'],columns='pliegue',values='peso')
weights_pivot['media']=weights_pivot.mean(axis=1)
print('PESOS EM POR MODELO Y PLIEGUE (y media):')
display(weights_pivot)

folds_table=pd.DataFrame(fold_rows)
reference_oof=macro_f1(y_development,oof['R'].argmax(axis=1))
reference_gap=np.abs(folds_table.f1_R.to_numpy()-REF_FOLD_F1)
print('Referencia R recalculada por pliegue:',folds_table.f1_R.round(6).tolist(),'| OOF',round(reference_oof,6))
print('Max |diferencia| con la referencia medida por pliegue:',float(reference_gap.max()),'| OOF:',abs(reference_oof-REF_OOF_F1))
if reference_gap.max()>REF_TOLERANCE or abs(reference_oof-REF_OOF_F1)>REF_TOLERANCE:
    raise RuntimeError('DETENER: la referencia R recalculada no coincide con la medida (tolerancia 1e-6).')
print('REFERENCIA R VERIFICADA (tolerancia 1e-6).')

labels={'R':'R (car 4, pal 3)','O6':'O6 (car 6, pal 3)','O7':'O7 (car 7, pal 3)','E':'E (mezcla EM car 3-6, pal 2-4)'}
result_rows=[]
for arm in ARMS:
    per_fold=folds_table[f'f1_{arm}'].to_numpy()
    differences=per_fold-folds_table.f1_R.to_numpy()
    result_rows.append(dict(brazo=arm,descripcion=labels[arm],f1_macro_oof=macro_f1(y_development,oof[arm].argmax(axis=1)),**{f'pliegue_{number}':value for number,value in enumerate(per_fold,1)},f1_pliegues_std=float(per_fold.std(ddof=1)),diferencia_pareada_media=float(differences.mean()),diferencia_pareada_std=float(differences.std(ddof=1))))
results=pd.DataFrame(result_rows)
print('TABLA: F1 macro OOF sin calibrar, F1 por pliegue, desviacion por pliegue y diferencia pareada contra R (ddof=1)')
print(results.to_string(index=False))
display(folds_table)
print('CV segundos:',round(time.perf_counter()-cv_started,1))


pliegue 1: R 0.440843 | O6 0.437601 | O7 0.434074 | E 0.440561 | pesos car [0.002, 0.02, 0.146, 0.833] pal [0.07, 0.286, 0.644] | 78 s


pliegue 2: R 0.432485 | O6 0.430954 | O7 0.429763 | E 0.433095 | pesos car [0.002, 0.022, 0.146, 0.83] pal [0.07, 0.303, 0.627] | 76 s


pliegue 3: R 0.440697 | O6 0.433738 | O7 0.434430 | E 0.435727 | pesos car [0.002, 0.021, 0.142, 0.835] pal [0.064, 0.29, 0.646] | 75 s


pliegue 4: R 0.447354 | O6 0.438003 | O7 0.439882 | E 0.440172 | pesos car [0.002, 0.022, 0.145, 0.831] pal [0.072, 0.287, 0.64] | 76 s


pliegue 5: R 0.439014 | O6 0.433266 | O7 0.433308 | E 0.437371 | pesos car [0.002, 0.021, 0.145, 0.832] pal [0.074, 0.285, 0.641] | 79 s


...

PESOS EM POR MODELO Y PLIEGUE (y media):


pliegue             1         2         3         4         5     media
nivel orden                                                            
char  3      0.001711  0.002112  0.002123  0.002153  0.002062  0.002032
      4      0.019650  0.021749  0.020960  0.021636  0.021314  0.021062
      5      0.145724  0.145753  0.142371  0.145064  0.144873  0.144757
      6      0.832915  0.830386  0.834546  0.831147  0.831751  0.832149
word  2      0.070319  0.070178  0.064395  0.072327  0.073678  0.070179
      3      0.285764  0.302967  0.289859  0.287462  0.285053  0.290221
      4      0.643917  0.626856  0.645746  0.640211  0.641268  0.639600

Referencia R recalculada por pliegue: [0.440843, 0.432485, 0.440697, 0.447354, 0.439014] | OOF 0.440081
Max |diferencia| con la referencia medida por pliegue: 4.232924309954811e-07 | OOF: 2.786557278255941e-07
REFERENCIA R VERIFICADA (tolerancia 1e-6).
TABLA: F1 macro OOF sin calibrar, F1 por pliegue, desviacion por pliegue y diferencia pareada contra R (ddof=1)
brazo                    descripcion  f1_macro_oof  pliegue_1  pliegue_2  pliegue_3  pliegue_4  pliegue_5  f1_pliegues_std  diferencia_pareada_media  diferencia_pareada_std
    R               R (car 4, pal 3)      0.440081   0.440843   0.432485   0.440697   0.447354   0.439014         0.005308                  0.000000                0.000000
   O6              O6 (car 6, pal 3)      0.434686   0.437601   0.430954   0.433738   0.438003   0.433266         0.003014                 -0.005366                0.003073
   O7              O7 (car 7, pal 3)      0.434295   0.434074   0.429763   0.434430   0.439882   0.433308         0.

,pliegue,f1_R,f1_O6,f1_O7,f1_E,segundos,segundos_em,filas_validacion_interna
0,1,0.440843,0.437601,0.434074,0.440561,77.851399,17.564214,10886
1,2,0.432485,0.430954,0.429763,0.433095,76.021420,17.885812,10638
2,3,0.440697,0.433738,0.434430,0.435727,75.244085,16.820186,10424
3,4,0.447354,0.438003,0.439882,0.440172,75.843916,17.633566,10434
4,5,0.439014,0.433266,0.433308,0.437371,78.550933,18.066369,10446


CV segundos: 383.8


## Decision

In [7]:
tie_order={'O6':0,'O7':1,'E':2}
candidates=results.loc[results.brazo!='R'].copy()
candidates['desempate']=candidates.brazo.map(tie_order)
ranked=candidates.sort_values(['diferencia_pareada_media','desempate'],ascending=[False,True])
accepted=ranked.loc[(ranked.diferencia_pareada_media>0)&(ranked.diferencia_pareada_media>ranked.diferencia_pareada_std)]
PASA=not accepted.empty
selected_arm=accepted.iloc[0].brazo if PASA else None
best_arm=ranked.iloc[0].brazo
for _,row in ranked.sort_values('desempate').iterrows():
    print(f'{row.brazo}: diferencia pareada media {row.diferencia_pareada_media:+.6f} | desviacion estandar {row.diferencia_pareada_std:.6f} | {"acepta" if row.brazo in set(accepted.brazo) else "no acepta"}')
print('Regla fijada: media >0 y > desviacion estandar muestral; desempate O6, O7, E.')
print('PASA' if PASA else 'NO PASA')
print('Brazo elegido:',selected_arm if PASA else 'ninguno','| mejor brazo descriptivo (mayor diferencia media):',best_arm)
if not PASA:
    print('NO PASA: holdout y CSV se omiten explicitamente; no se creara age_submission_v8_ordenes_y_ensamble.csv.')


O6: diferencia pareada media -0.005366 | desviacion estandar 0.003073 | no acepta
O7: diferencia pareada media -0.005787 | desviacion estandar 0.001832 | no acepta
E: diferencia pareada media -0.002693 | desviacion estandar 0.003285 | no acepta
Regla fijada: media >0 y > desviacion estandar muestral; desempate O6, O7, E.
NO PASA
Brazo elegido: ninguno | mejor brazo descriptivo (mayor diferencia media): E
NO PASA: holdout y CSV se omiten explicitamente; no se creara age_submission_v8_ordenes_y_ensamble.csv.


## Holdout (solo si PASA)

In [8]:
if PASA:
    assert 'holdout_f1' not in globals(),'Holdout ya evaluado; no reutilizar.'
    started=time.perf_counter()
    texts_holdout=train.normalized.iloc[idx_holdout].to_numpy()
    assert not {text.lower() for text in texts_development}&{text.lower() for text in texts_holdout}
    holdout_scores,holdout_info=arm_scores(texts_development,y_development,groups_development,texts_holdout,seed=SEED+90)
    holdout_pred=holdout_scores[selected_arm].argmax(axis=1)
    y_holdout=y_all[idx_holdout]
    holdout_f1=macro_f1(y_holdout,holdout_pred)
    print('F1 macro holdout v7_lower:',holdout_f1,'| brazo',selected_arm)
    if selected_arm=='E':
        print('Pesos EM (desarrollo completo):',{level:np.round(weights,4).tolist() for level,weights in holdout_info['weights'].items()})
    display(pd.DataFrame(classification_report(y_holdout,holdout_pred,labels=np.arange(6),target_names=CLASSES,output_dict=True,zero_division=0)).T.loc[CLASSES,['precision','recall','f1-score','support']])
    print('Holdout segundos:',round(time.perf_counter()-started,1),'; evaluacion unica. No comparar directamente con el holdout de v6 ni con Kaggle.')
else:
    print('HOLDOUT OMITIDO: NO PASA; no se entrena ni evalua en holdout.')


HOLDOUT OMITIDO: NO PASA; no se entrena ni evalua en holdout.


## CSV (solo si PASA)

In [9]:
if PASA:
    verify_csv(DATA_DIR/'age_test.csv',['id','text'],';',True)
    test=pd.read_csv(DATA_DIR/'age_test.csv',sep=';',encoding='utf-8-sig',dtype=str,keep_default_na=False)
    assert list(test.columns)==['id','text'] and test.id.is_unique and test.id.ne('').all()
    test_texts=test.text.fillna('').str.strip().map(html.unescape).map(normalize_text).to_numpy()
    test_scores,_=arm_scores(train.normalized.to_numpy(),y_all,group_ids,test_texts,seed=SEED+99)
    submission=pd.DataFrame({'id':test.id,'age_range':CLASSES[test_scores[selected_arm].argmax(axis=1)]})
    assert list(submission.columns)==reference_columns==['id','age_range']
    assert len(submission)==len(test)==112892 and submission.id.is_unique and submission.id.tolist()==test.id.tolist()
    assert submission.age_range.isin(CLASSES).all()
    with SUB_PATH.open('x',encoding='utf-8',newline='') as handle:
        submission.to_csv(handle,index=False,sep=',')
    print('CSV VERIFICADO:',SUB_PATH.name,len(submission),'filas; coma y sin BOM.')
else:
    assert not SUB_PATH.exists()
    print('CSV OMITIDO: NO PASA; no se carga el test para predecir ni se genera salida.')


CSV OMITIDO: NO PASA; no se carga el test para predecir ni se genera salida.


## F1 por clase y matriz de confusion del mejor brazo

In [10]:
best_predictions=oof[best_arm].argmax(axis=1)
per_class=pd.DataFrame({
    'f1_R':f1_score(y_development,oof['R'].argmax(axis=1),labels=np.arange(6),average=None,zero_division=0),
    f'f1_{best_arm}':f1_score(y_development,best_predictions,labels=np.arange(6),average=None,zero_division=0),
    'soporte_desarrollo':np.bincount(y_development,minlength=6),
},index=CLASSES)
per_class['diferencia']=per_class[f'f1_{best_arm}']-per_class.f1_R
print('F1 por clase (OOF) de R y del mejor brazo:',best_arm)
display(per_class)
matrix=confusion_matrix(y_development,best_predictions,labels=np.arange(6))
print('Matriz de confusion OOF del mejor brazo:',best_arm)
display(pd.DataFrame(matrix,index=[f'real {name}' for name in CLASSES],columns=[f'pred {name}' for name in CLASSES]))
adjacent_share=sum(matrix[i,j] for i in range(6) for j in range(6) if abs(i-j)==1)/max(1,matrix.sum()-matrix.trace())
print(f'HIPOTESIS (no verificada): {adjacent_share:.1%} de los errores caen en rangos adyacentes; las clases centrales (25-34, 35-49) siguen siendo las de menor F1, lo que sugeriria que su estilo se solapa con el de sus vecinas mas que faltar contexto de caracteres. Requiere analisis cualitativo antes de afirmarlo.')


F1 por clase (OOF) de R y del mejor brazo: E


,f1_R,f1_E,soporte_desarrollo,diferencia
13-17,0.502141,0.502390,22792,0.000249
18-24,0.381955,0.364479,22552,-0.017476
25-34,0.343792,0.341540,22405,-0.002251
35-49,0.370839,0.376862,22168,0.006022
50-64,0.495927,0.490963,22023,-0.004964
65-xx,0.545829,0.547876,21047,0.002047


Matriz de confusion OOF del mejor brazo: E


,pred 13-17,pred 18-24,pred 25-34,pred 35-49,pred 50-64,pred 65-xx
real 13-17,11245,5213,2735,1570,1170,859
real 18-24,5098,8061,4177,2265,1661,1290
real 25-34,2769,4167,7470,3420,2612,1967
real 35-49,1372,2153,3640,8033,3883,3087
real 50-64,813,1123,1970,2946,11422,3749
real 65-xx,677,964,1346,2229,3758,12073


HIPOTESIS (no verificada): 53.6% de los errores caen en rangos adyacentes; las clases centrales (25-34, 35-49) siguen siendo las de menor F1, lo que sugeriria que su estilo se solapa con el de sus vecinas mas que faltar contexto de caracteres. Requiere analisis cualitativo antes de afirmarlo.


## Resumen

In [11]:
best_row=results.loc[results.brazo==best_arm].iloc[0]
history.loc[history.intento=='v8 ordenes y ensamble',['f1_oof','estado']]=[f'{best_row.f1_macro_oof:.6f} (mejor brazo {best_arm})','PASA' if PASA else 'NO PASA']
display(history)
arm_lines='; '.join(f'{row.brazo}: OOF {row.f1_macro_oof:.6f}, diferencia media {row.diferencia_pareada_media:+.6f} (desviacion {row.diferencia_pareada_std:.6f})' for _,row in results.loc[results.brazo!='R'].iterrows())
mean_weights='; '.join(f'{level} orden {order}: {weight:.3f}' for (level,order),weight in weights_pivot['media'].items())
summary_text=f'''## Resumen final: ordenes altos y ensamble EM

Resultado: **{'PASA' if PASA else 'NO PASA'}**. Referencia R recalculada: F1 OOF {reference_oof:.6f}. {arm_lines}. Pesos EM medios de E: {mean_weights}. {'Holdout evaluado una sola vez con el brazo ' + selected_arm + ': F1 = ' + format(holdout_f1, '.6f') + '; CSV generado con escritura exclusiva.' if PASA else 'Holdout y CSV omitidos explicitamente por la regla.'}

**Ideas.** (1) Ordenes de caracteres 6 y 7: historiales mas largos para capturar secuencias que el orden 4 corta, con Kneser-Ney modificado respaldando en ordenes bajos los contextos escasos. (2) Ensamble E: mezcla lineal por token de ModifiedKN de varios ordenes, con pesos aprendidos por EM en una validacion interna del ajuste; dos mezclas (caracteres 3-6 y palabras 2-4) porque una mezcla por token solo combina modelos que predicen el mismo tipo de simbolo; los niveles se combinan con el 0,75/0,25 de R. Todo sigue siendo un modelo de lenguaje propio y generativo por clase, sin clasificador discriminativo sobre scores.

**Optimismo de seleccion:** comparar tres brazos (O6, O7, E) y elegir uno introduce optimismo de seleccion. La regla (diferencia pareada media > 0 y mayor que su desviacion estandar, ddof=1) es una heuristica, no una prueba formal.

**Comparabilidad:** la particion v7_lower no es directamente comparable con el holdout de v6; los scores publicos de Kaggle no son comparables con holdout. Referencias confirmadas: v5 calibrado 0,43847, v6 sin calibrar 0,44106, v6 calibrado 0,44353 (particiones distintas) y referencia en v7_lower 0,440081; el jerarquico no paso (beta 10, diferencia pareada -0,0255).

NPZ v7_lower reutilizado sin regenerar, con hashes de archivo, train y grupos verificados y cero fuga en minusculas. Normalizacion A de v6. Sin calibracion de sesgos, sin commit, push ni envios a Kaggle. Entorno .venv-v6, kernel tweetage-v6-kn.
'''
display(Markdown(summary_text))
assert INPUT_HASHES=={name:hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest() for name in INPUT_HASHES}
assert PROTECTED_HASHES=={name:hashlib.sha256((OUT_DIR/name).read_bytes()).hexdigest() for name in PROTECTED_HASHES}
print('HASHES DE TODOS LOS ARCHIVOS PREEXISTENTES PROTEGIDOS: SIN CAMBIOS.')


,intento,f1_oof,estado
0,v5 KN (particion con fuga),"0,43847 (calibrado)","Kaggle 0,45643; holdout inflado"
1,v6 KN modificado (agrupada),"0,44106 sin calibrar / 0,44353 calibrado","Valida publicada; Kaggle 0,45737"
2,Referencia en v7_lower,"0,440081",Referencia de v7 y v8
3,v7 ordinal,sin salida guardada,NO PASA
4,v7 descuento con concentracion,"0,440875",NO PASA
5,v7 normalizacion (A/B/C),"0,440081 / 0,439709 / 0,436893",NO PASA
6,v7 hiperparametros (mejor C7),"0,437677",NO PASA
7,v7 LM neuronal GRU,"pliegue 1: 0,411131",NO PASA EN ETAPA 1
8,v8 Naive Bayes,"0,376727",NO PASA
9,v8 PPM-C,"0,421730",NO PASA


## Resumen final: ordenes altos y ensamble EM

Resultado: **NO PASA**. Referencia R recalculada: F1 OOF 0.440081. O6: OOF 0.434686, diferencia media -0.005366 (desviacion 0.003073); O7: OOF 0.434295, diferencia media -0.005787 (desviacion 0.001832); E: OOF 0.437352, diferencia media -0.002693 (desviacion 0.003285). Pesos EM medios de E: char orden 3: 0.002; char orden 4: 0.021; char orden 5: 0.145; char orden 6: 0.832; word orden 2: 0.070; word orden 3: 0.290; word orden 4: 0.640. Holdout y CSV omitidos explicitamente por la regla.

**Ideas.** (1) Ordenes de caracteres 6 y 7: historiales mas largos para capturar secuencias que el orden 4 corta, con Kneser-Ney modificado respaldando en ordenes bajos los contextos escasos. (2) Ensamble E: mezcla lineal por token de ModifiedKN de varios ordenes, con pesos aprendidos por EM en una validacion interna del ajuste; dos mezclas (caracteres 3-6 y palabras 2-4) porque una mezcla por token solo combina modelos que predicen el mismo tipo de simbolo; los niveles se combinan con el 0,75/0,25 de R. Todo sigue siendo un modelo de lenguaje propio y generativo por clase, sin clasificador discriminativo sobre scores.

**Optimismo de seleccion:** comparar tres brazos (O6, O7, E) y elegir uno introduce optimismo de seleccion. La regla (diferencia pareada media > 0 y mayor que su desviacion estandar, ddof=1) es una heuristica, no una prueba formal.

**Comparabilidad:** la particion v7_lower no es directamente comparable con el holdout de v6; los scores publicos de Kaggle no son comparables con holdout. Referencias confirmadas: v5 calibrado 0,43847, v6 sin calibrar 0,44106, v6 calibrado 0,44353 (particiones distintas) y referencia en v7_lower 0,440081; el jerarquico no paso (beta 10, diferencia pareada -0,0255).

NPZ v7_lower reutilizado sin regenerar, con hashes de archivo, train y grupos verificados y cero fuga en minusculas. Normalizacion A de v6. Sin calibracion de sesgos, sin commit, push ni envios a Kaggle. Entorno .venv-v6, kernel tweetage-v6-kn.


HASHES DE TODOS LOS ARCHIVOS PREEXISTENTES PROTEGIDOS: SIN CAMBIOS.
